### Preparing the Dataset

In [ ]:
import pandas as pd  # pandas for data manipulation and DataFrame operations
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv('demo_dataset.csv')

display(df)

### One-Hot Encoding
One-hot encoding takes a categorical feature and converts it into a set of new binary features, where each binary feature corresponds to one possible category value. This process creates a set of indicator columns that hold 1 or 0, indicating the presence or absence of a particular category in each row.

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) # ignore unseen categories, return dense array
encoded = encoder.fit_transform(df[['protocol']]) # fit and transform the protocol column
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['protocol'])) # name the new binary columns

df = pd.concat([df.drop('protocol', axis=1), encoded_df], axis=1) # replace original column with encoded ones

### Handling Skewed Data
When a feature is skewed, its values are unevenly distributed, often with most observations clustered near one end and a few extreme values stretching out the distribution. Such skew can affect the performance of machine learning models, especially those sensitive to outliers or that assume more uniform or normal-like data distributions.

In [ ]:
import numpy as np

df["bytes_transferred"] = pd.to_numeric(df["bytes_transferred"], errors='coerce') # convert strings to NaN
df["bytes_transferred"] = np.log1p(df["bytes_transferred"]) # log transform, +1 to avoid log(0)

### Data Splitting
Data splitting involves dividing a dataset into three distinct subsets—training, validation, and testing—to ensure reliable model evaluation. 80/20 has a statistical basis called the Pareto principle

In [49]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df.drop("threat_level", axis=1)
y = df["threat_level"]

# Initial split: 80% training, 20% testing, random_state is a seed to make the split deterministic
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1337)

# Second split: from the 80% training portion, allocate 60% for final training and 20% for validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=1337)